# ✋ HAND LAB — 21개 점으로 만드는 가위바위보 인식기

> **[26년 3기] NPU 활용 온디바이스 AI 프로그래밍** · 특별 세션 · **CPU 런타임으로 충분**

---

## 🔑 핵심 메시지

> **"모델을 바꾸기 전에 좌표계를 바꿔라."**
>
> 오늘 우리는 같은 문제를 세 번 풉니다 — ① 손으로 짠 규칙, ② 날좌표를 먹인 신경망, ③ 정규화 좌표를 먹인 신경망.
> 결과를 미리 말하면: **학습 모델이 규칙에게 집니다.** 그리고 전처리 딱 세 줄이 그 패배를 만점으로 뒤집습니다.
> 특징 공학(feature engineering)이 왜 죽지 않았는지, 숫자로 확인하는 랩입니다.

## 📋 실습 로드맵

| Part | 주제 | 도구 | 재현성 |
|---|---|---|---|
| 1 | 21개 랜드마크 해부 | MediaPipe Tasks | 📊 |
| 2 | 손가락 굽힘 = 각도 판정 | 순수 numpy (POSE LAB의 `joint_angle` 재사용) | ✅ |
| 3 | 규칙 기반 가위바위보 | seeded 합성 손 300개 | ✅ **96.00%** |
| 4 | 학습 기반 — 3막의 반전 | 순수 numpy MLP | ✅ **73.00% → 100.00%** |
| 5 | 내 손으로 검증 | 사진 업로드 | 📊 |

## ⚙️ 실행 환경
- 런타임: **CPU** · 전체 실행 시간 📊 약 8~12분 (MLP 학습 포함)


---
# Part 0 · 환경 설정

In [ ]:
# [0-1] MediaPipe 설치 + 손 랜드마커 모델 다운로드
!pip install -q mediapipe
!curl -sL -o hand_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!ls -lh hand_landmarker.task

import mediapipe as mp
print(f"✅ mediapipe {mp.__version__}")

**📊 기대 출력** — 약 `7.5M` 크기 파일.

> 🧑‍🏫 **강사 노트**: 손 파이프라인도 POSE LAB에서 본 **2단 구조**입니다 — 손바닥 검출기(palm detector)가 가끔, 랜드마크 모델이 매 프레임. 시리즈 관통 원리("비싼 것은 드물게")를 한 문장으로 상기시키고 넘어가세요.

In [ ]:
# [0-2] 실습 이미지 + 공통 임포트
import numpy as np
import cv2
import matplotlib.pyplot as plt
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

!curl -sL -o hand_sample.jpg \
  https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg

image = cv2.cvtColor(cv2.imread("hand_sample.jpg"), cv2.COLOR_BGR2RGB)
print(f"이미지 크기: {image.shape}")
plt.figure(figsize=(6,5)); plt.imshow(image); plt.axis('off'); plt.show()

> 🧑‍🏫 **강사 노트**: 공식 샘플 이미지입니다. Part 5에서 학생 각자의 손 사진(가위/바위/보 각 1장)을 쓰므로, 수업 시작 시 미리 찍어두라고 안내하면 진행이 매끄럽습니다. 배경이 단순하고 손이 화면의 1/3 이상이면 인식률이 좋습니다.

---
# Part 1 · 21개 랜드마크 해부

MediaPipe Hands의 출력은 손목(0) + 손가락 5개 × 4관절 = **21개** 점입니다.

```
        8   12  16  20      ← TIP (손끝)
        |   |   |   |
        7   11  15  19      ← DIP
        |   |   |   |
        6   10  14  18      ← PIP
    4   |   |   |   |
     \  5   9   13  17      ← MCP (손가락 뿌리)
      3  \  |  /   /
       \  \ | /   /
        2  \|/   /
         \  |   /
          1 |  /
           \| /
            0               ← WRIST (손목)
```

인덱스 규칙: **엄지 1~4, 검지 5~8, 중지 9~12, 약지 13~16, 새끼 17~20** — 각 손가락은 MCP→PIP→DIP→TIP 순서. 이 규칙만 외우면 어떤 제스처 로직도 짤 수 있습니다.

In [ ]:
# [1-1] 손 랜드마커 로드 + 추론
base = mp_python.BaseOptions(model_asset_path="hand_landmarker.task")
opts = vision.HandLandmarkerOptions(base_options=base, num_hands=2)
hander = vision.HandLandmarker.create_from_options(opts)

mp_img = mp.Image.create_from_file("hand_sample.jpg")

import time
t0 = time.perf_counter()
result = hander.detect(mp_img)
t_ms = (time.perf_counter()-t0)*1000

print(f"검출된 손      : {len(result.hand_landmarks)}개")
for hd in result.handedness:
    print(f"  → {hd[0].category_name} (신뢰도 {hd[0].score:.3f})")
print(f"랜드마크/손    : {len(result.hand_landmarks[0])}개")   # ✅ 21
print(f"추론 시간(CPU) : {t_ms:.0f} ms  📊 (초회는 초기화 포함으로 김, 이후 20~80ms)")

> 💡 `handedness`가 Left/Right를 알려줍니다 — 단, **카메라에 비친 좌우**라서 셀피(전면) 카메라에서는 거울 반전을 주의해야 합니다. 리포트 실험 ③의 소재입니다.

In [ ]:
# [1-2] 스켈레톤 그리기 — 손가락별 색 구분
lms = result.hand_landmarks[0]
h, w = image.shape[:2]

FINGER_IDX = {"thumb":[1,2,3,4], "index":[5,6,7,8], "middle":[9,10,11,12],
              "ring":[13,14,15,16], "pinky":[17,18,19,20]}
COLORS = {"thumb":(255,180,84), "index":(77,201,255), "middle":(57,217,138),
          "ring":(185,138,255), "pinky":(255,93,93)}

canvas = image.copy()
for name, idxs in FINGER_IDX.items():
    chain = [0] + idxs                      # 손목에서 시작
    for a, b in zip(chain[:-1], chain[1:]):
        pa = (int(lms[a].x*w), int(lms[a].y*h))
        pb = (int(lms[b].x*w), int(lms[b].y*h))
        cv2.line(canvas, pa, pb, COLORS[name], 3)
for i in range(21):
    p = (int(lms[i].x*w), int(lms[i].y*h))
    cv2.circle(canvas, p, 5, (255,255,255), -1)
    cv2.putText(canvas, str(i), (p[0]+6, p[1]-4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (230,230,230), 1)

plt.figure(figsize=(7,6)); plt.imshow(canvas); plt.axis('off')
plt.title("21 landmarks — 손가락별 색, 번호는 인덱스"); plt.show()

---
# Part 2 · 손가락 굽힘 = 각도 판정

POSE LAB에서 만든 `joint_angle`을 **그대로 재사용**합니다 — 무릎이든 손가락이든, 세 점의 각도는 같은 내적입니다.

**굽힘 판정 규칙**: 각 손가락의 **MCP–PIP–TIP** 세 점 각도가 `140°`보다 크면 "펴짐(extended)".

| 상태 | MCP-PIP-TIP 각도 | 판정 |
|---|---|---|
| 완전히 폄 | **180.0000°** | extended ✓ |
| 깊이 굽힘 (curl 0.9) | **60.7186°** | curled ✗ |

(위 두 값은 Part 3의 합성 손에서 나온 ✅ 완전 결정적 수치입니다)

In [ ]:
# [2-1] joint_angle — POSE LAB과 동일 함수 (시리즈 공용 부품)
def joint_angle(a, b, c):
    """b를 꼭짓점으로 하는 ∠abc (도)"""
    a, b, c = map(np.asarray, (a, b, c))
    v1, v2 = a - b, c - b
    cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return float(np.degrees(np.arccos(np.clip(cos, -1, 1))))

FI = {"index": 5, "middle": 9, "ring": 13, "pinky": 17}   # 각 손가락 MCP 인덱스

def finger_angle(pts, name):
    """pts: (21,2) 배열 — MCP-PIP-TIP 각도"""
    m = FI[name]
    return joint_angle(pts[m], pts[m+1], pts[m+3])

def is_extended(pts, name, thr=140.0):
    return finger_angle(pts, name) > thr

In [ ]:
# [2-2] 실제 사진의 손에 적용
pts_real = np.array([[lm.x, lm.y] for lm in lms])   # 21×2 정규화 좌표

print(f"{'손가락':<8} {'각도':>10} {'판정':>10}")
for name in FI:
    a = finger_angle(pts_real, name)
    print(f"{name:<8} {a:>9.1f}° {'extended' if a>140 else 'curled':>10}")
print()
print("📊 사진 속 손 모양과 판정이 일치하는지 눈으로 확인하세요")

---
# Part 3 · 규칙 기반 가위바위보 — 정답을 아는 실험실

실제 손 사진으로 분류기를 평가하려면 라벨링이 필요합니다. 대신 우리는 **합성 손 생성기**를 만듭니다 — 굽힘(curl) 파라미터로 손을 "제조"하므로 정답을 100% 알고, seeded라서 ✅ 완전 재현됩니다. (POSE LAB의 합성 스쿼트 신호와 같은 전략: **로직 검증은 결정적 환경에서 먼저**)

In [ ]:
# [3-1] 합성 손 생성기 — curl 0(폄)~1(굽힘)로 손을 제조
FINGERS = {"index":  {"fan": -15, "mcp_r": 0.50, "segs": [0.34, 0.24, 0.17]},
           "middle": {"fan":   0, "mcp_r": 0.52, "segs": [0.37, 0.26, 0.18]},
           "ring":   {"fan":  15, "mcp_r": 0.50, "segs": [0.34, 0.24, 0.17]},
           "pinky":  {"fan":  32, "mcp_r": 0.46, "segs": [0.27, 0.19, 0.15]}}
CURL_PER_JOINT = 95.0     # curl=1일 때 관절당 굽힘(도)

def make_hand(curls, rot_deg=0.0, scale=1.0, tx=0.0, ty=0.0, noise=0.0, rng=None):
    """curls: {손가락: 0~1} → 21×2 랜드마크 (MediaPipe 인덱스 배치)"""
    pts = np.zeros((21, 2))
    # 엄지(1~4): 모든 제스처에서 반굽힘 고정 (실전 단순화 — 리포트 ①에서 확장)
    d = np.deg2rad(-55); p = np.zeros(2); idx = 1
    for L in [0.30, 0.22, 0.18]:
        p = p + L*np.array([np.sin(d), np.cos(d)]); pts[idx] = p; idx += 1
        d += np.deg2rad(20)
    pts[4] = pts[3] + 0.10*np.array([np.sin(d), np.cos(d)])
    # 네 손가락(5~20): MCP → PIP → DIP → TIP
    base = 5
    for name, spec in FINGERS.items():
        fan = np.deg2rad(spec["fan"])
        mcp = spec["mcp_r"]*np.array([np.sin(fan), np.cos(fan)])
        pts[base] = mcp
        dd = fan; p = mcp
        for k, L in enumerate(spec["segs"]):
            if k > 0: dd += np.deg2rad(curls[name]*CURL_PER_JOINT)
            p = p + L*np.array([np.sin(dd), np.cos(dd)])
            pts[base+1+k] = p
        base += 4
    # 전역 변환: 회전 → 스케일 → 평행이동 → 노이즈 (실전의 다양성 흉내)
    r = np.deg2rad(rot_deg)
    R = np.array([[np.cos(r), -np.sin(r)], [np.sin(r), np.cos(r)]])
    pts = (pts @ R.T) * scale + np.array([tx, ty])
    if noise > 0 and rng is not None:
        pts = pts + rng.normal(0, noise*scale, pts.shape)
    return pts

# 캐논 3종 확인 — 각도가 Part 2 표의 ✅ 값과 일치해야 합니다
canon_paper    = make_hand({n: 0.0 for n in FI})
canon_rock     = make_hand({n: 0.9 for n in FI})
canon_scissors = make_hand({"index":0.0, "middle":0.0, "ring":0.9, "pinky":0.9})
print(f"보 검지    : {finger_angle(canon_paper,'index'):.4f}°")
print(f"바위 검지  : {finger_angle(canon_rock,'index'):.4f}°")
print(f"가위 약지  : {finger_angle(canon_scissors,'ring'):.4f}°")

**✅ 기대 출력** (완전 결정적)
```
보 검지    : 180.0000°
바위 검지  : 60.7186°
가위 약지  : 60.7186°
```

In [ ]:
# [3-2] 규칙 분류기 — if문 4개가 전부
def rule_classify(pts, thr=140.0):
    ext = {n: is_extended(pts, n, thr) for n in FI}
    n_ext = sum(ext.values())
    if n_ext == 0:                                        return "rock"
    if n_ext == 4:                                        return "paper"
    if n_ext == 2 and ext["index"] and ext["middle"]:     return "scissors"
    return "unknown"                                      # 애매하면 기권

for name, hand in [("보", canon_paper), ("바위", canon_rock), ("가위", canon_scissors)]:
    print(f"{name} → {rule_classify(hand)}")

In [ ]:
# [3-3] seeded 데이터셋 300개 — 규칙의 성적표
rng = np.random.default_rng(42)                          # seeded!
LABELS = ["rock", "paper", "scissors"]

def sample_curls(label, rng):
    if label == "rock":  return {n: rng.uniform(0.72, 1.0)  for n in FI}
    if label == "paper": return {n: rng.uniform(0.0,  0.16) for n in FI}
    return {"index": rng.uniform(0.0, 0.16), "middle": rng.uniform(0.0, 0.16),
            "ring":  rng.uniform(0.72, 1.0), "pinky":  rng.uniform(0.72, 1.0)}

X, y = [], []
for i in range(300):
    lab = LABELS[i % 3]
    X.append(make_hand(sample_curls(lab, rng),
                       rot_deg=rng.uniform(-15, 15), scale=rng.uniform(0.6, 1.8),
                       tx=rng.uniform(-4, 4), ty=rng.uniform(-4, 4),
                       noise=0.03, rng=rng))              # 랜드마크 지터
    y.append(lab)
X = np.array(X)

rule_acc = np.mean([rule_classify(p) == l for p, l in zip(X, y)])
print(f"규칙 기반 정확도: {rule_acc*100:.2f}%   (300샘플, 위치·크기·회전·노이즈 랜덤)")

**✅ 기대 출력** (seed=42 고정)
```
규칙 기반 정확도: 96.00%
```

각도는 평행이동·스케일에 **원래 불변**이므로, 손이 화면 어디에 어떤 크기로 있어도 규칙은 흔들리지 않습니다. 96%를 깎아먹은 건 오직 관절 노이즈뿐 — 어떤 샘플이 틀렸는지 궁금하면 `unknown` 판정을 세어보세요 (리포트 ②의 출발점).

---
# Part 4 · 학습 기반 — 3막의 반전 🎬

"규칙은 구식이고 학습이 미래다" — 정말일까요? 42차원 벡터(21점 × 2좌표)를 먹는 작은 MLP를 순수 numpy로 학습시켜 봅니다. 프레임워크 없이, 역전파까지 30줄.

In [ ]:
# [4-1] 순수 numpy MLP (42 → 32 → 3) — seeded 결정적 학습
yv = np.array([LABELS.index(l) for l in y])
perm = np.random.default_rng(7).permutation(300)
tr, te = perm[:200], perm[200:]                          # 200 학습 / 100 평가

def train_mlp(feats, seed=0, epochs=400, lr=0.5):
    rs = np.random.default_rng(seed)
    W1 = rs.normal(0, .3, (feats.shape[1], 32)); b1 = np.zeros(32)
    W2 = rs.normal(0, .3, (32, 3));              b2 = np.zeros(3)
    Xtr, Xte, Ytr = feats[tr], feats[te], np.eye(3)[yv[tr]]
    for _ in range(epochs):
        H = np.tanh(Xtr@W1 + b1)                          # forward
        Z = H@W2 + b2; Z -= Z.max(1, keepdims=True)
        P = np.exp(Z); P /= P.sum(1, keepdims=True)       # softmax
        G = (P - Ytr)/len(Xtr)                            # backward
        gW2, gb2 = H.T@G, G.sum(0)
        GH = (G@W2.T)*(1 - H**2)
        gW1, gb1 = Xtr.T@GH, GH.sum(0)
        W1 -= lr*gW1; b1 -= lr*gb1; W2 -= lr*gW2; b2 -= lr*gb2
    pred = (np.tanh(Xte@W1+b1)@W2 + b2).argmax(1)
    return float(np.mean(pred == yv[te]))

# ── 2막: 날좌표 그대로 ──
acc_raw = train_mlp(X.reshape(300, -1))
print(f"MLP (raw 좌표): {acc_raw*100:.2f}%")

**✅ 기대 출력**
```
MLP (raw 좌표): 73.00%
```

### 🎬 학습이 규칙에게 졌습니다 — 73% vs 96%

이유: 같은 "가위"라도 손의 **위치·크기가 다르면 날좌표는 완전히 다른 벡터**입니다. 42차원 공간에서 같은 제스처가 사방에 흩어져 있으니, 200개 샘플로는 그 흩어짐을 다 외울 수 없습니다. 규칙 기반의 각도는 처음부터 그 흩어짐에 면역이었고요.

In [ ]:
# [4-2] 3막: 정규화 세 줄 — 손목 원점 + 손 크기 나눗셈
def normalize(pts):
    q = pts - pts[0]                          # ① 손목을 원점으로 (위치 불변)
    s = np.max(np.linalg.norm(q, axis=1))     # ② 손 크기 측정
    return q / s                              # ③ 크기 1로 정규화 (스케일 불변)

X_norm = np.array([normalize(p) for p in X]).reshape(300, -1)
acc_norm = train_mlp(X_norm)

print("┌──────────────────────────────────────┐")
print(f"│ 규칙 기반            : {rule_acc*100:6.2f}%      │")
print(f"│ MLP  (raw 좌표)      : {acc_raw*100:6.2f}%      │")
print(f"│ MLP  (정규화 좌표)   : {acc_norm*100:6.2f}%      │")
print("└──────────────────────────────────────┘")

**✅ 기대 출력** (seed 고정 — 정확히 이 값)
```
│ 규칙 기반            :  96.00%      │
│ MLP  (raw 좌표)      :  73.00%      │
│ MLP  (정규화 좌표)   : 100.00%      │
```

### 📌 3막 정리

| 막 | 방법 | 정확도 | 교훈 |
|---|---|---|---|
| 1막 | 규칙 (각도) | 96% | 좋은 특징은 불변성을 공짜로 준다 |
| 2막 | MLP + 날좌표 | 73% | 모델은 죄가 없다 — 좌표계가 문제 |
| 3막 | MLP + 정규화 | 100% | **전처리 세 줄 = +27%p** |

> 같은 모델, 같은 데이터, 같은 seed — 바뀐 건 입력의 좌표계뿐입니다. Day 2에서 배운 캘리브레이션("어떤 데이터로 스케일을 정하느냐가 정확도를 좌우")과 같은 계열의 진실: **모델 밖의 결정이 모델 안의 성능을 지배합니다.**
> 노이즈를 키우면(리포트 ②) 규칙은 내려가고 raw MLP는 오히려 올라가는 역전 현상도 관찰됩니다 — 규칙의 경직성 vs 학습의 유연성 트레이드오프.

---
# Part 5 · 내 손으로 최종 검증

이제 진짜 손입니다. 가위/바위/보를 한 장씩 찍어 업로드하고, **규칙 분류기**를 그대로 적용합니다 — 합성에서 검증된 로직이 실전으로 이식되는 순간입니다.

In [ ]:
# [5-1] 사진 업로드 → MediaPipe → 규칙 분류
from google.colab import files
up = files.upload()                                      # 📊 여러 장 선택 가능

for fname in up:
    mp_i = mp.Image.create_from_file(fname)
    res = hander.detect(mp_i)
    if not res.hand_landmarks:
        print(f"{fname}: ❌ 손 미검출 (배경 단순화·손 크게 재촬영)")
        continue
    p = np.array([[lm.x, lm.y] for lm in res.hand_landmarks[0]])
    angles = {n: finger_angle(p, n) for n in FI}
    verdict = rule_classify(p)
    astr = " ".join(f"{n[:2]}:{a:.0f}°" for n, a in angles.items())
    print(f"{fname}: → {verdict:8s} | {astr}")
print()
print("📊 오판이 나오면 각도를 보세요 — thr=140° 근처에서 갈렸다면 임계값 문제,")
print("   전혀 다른 각도라면 랜드마크 자체가 틀린 것 (조명·배경·손 방향 확인)")

> 🧑‍🏫 **강사 노트 (양방향)**: 학생 실사진에서 **오판이 안 나올 수도, 자주 날 수도** 있습니다. 안 나오면 — 합성 검증의 승리로 마무리. 자주 나면 — 그 자리에서 각도 출력을 보며 원인 분리(임계값 vs 랜드마크)를 시연하세요. 특히 **손등/손바닥 방향, 카메라를 향한 손가락**(단축, foreshortening)이 2D 각도를 왜곡하는 사례가 나오면 최고의 토론 소재입니다 — "2D의 한계 → z좌표 활용"으로 리포트 ③과 연결됩니다.

---
# Part 6 · 리포트 과제 (3종 중 2종 선택)

### 실험 ① — 엄지를 살려라
현재 규칙은 엄지를 무시합니다. 엄지 판정(예: TIP(4)와 검지 MCP(5)의 거리, 또는 1-2-4 각도)을 추가해 **"좋아요(엄지만 폄)"** 제스처를 4번째 클래스로 확장하고, 합성 생성기에도 엄지 curl 파라미터를 넣어 정확도를 측정하세요. 엄지는 왜 다른 손가락과 같은 각도 규칙이 안 통할까요?

### 실험 ② — 노이즈 스트레스 테스트: 규칙 vs 학습의 역전
`noise`를 0.02 → 0.03 → 0.05 → 0.08로 키우며 세 분류기(규칙 / raw MLP / 정규화 MLP)의 정확도를 표로 기록하세요.
- 예고: 어느 지점부터 **규칙이 raw MLP에게 역전당합니다**. 경직된 임계값과 유연한 학습의 트레이드오프를 데이터로 논하세요.

### 실험 ③ — 3차원 구출 작전
MediaPipe는 z좌표도 줍니다. 카메라를 향해 손가락을 겨눈(단축된) 사진에서 2D 각도 판정이 실패하는 사례를 만들고, `joint_angle`을 3D 버전으로 바꿔 (x,y,z) 각도로 재판정해 개선 여부를 확인하세요.

---

## ✅ 체크포인트 — 오늘 확보해야 할 3가지
1. **캐논 각도 3종** `[3-1]` — 180.0000° / 60.7186° / 60.7186°
2. **3막 성적표** `[4-2]` — 규칙 96.00% · raw 73.00% · 정규화 100.00%
3. **내 손 판정 스크린샷** `[5-1]` — 가위바위보 3장 결과

> 🔗 **NPU 파이프라인과의 연결 (교육자 노트)**
> 오늘의 3막은 우리 과정의 두 지점과 공명합니다. 첫째, **정규화 = 캘리브레이션의 사촌** — Day 2 PTQ에서 "어떤 분포로 스케일을 잡느냐"가 INT8 정확도를 좌우했듯, 여기서는 좌표 스케일이 학습 성능을 좌우했습니다. 둘째, **규칙 기반 후처리의 가치** — 21점 랜드마크 모델은 NPU/DSP에 올리고 제스처 판정은 CPU의 if문 몇 개로 끝내는 구조는, YOLO+NMS 분리와 동일한 "모델은 가속기에, 로직은 호스트에" 패턴입니다. 엣지에서 학습 기반 제스처 헤드를 하나 더 올리는 것과 규칙 if문의 비용 차이를 생각해 보면, 96%짜리 규칙이 실무에서 사랑받는 이유가 보입니다.